# RF-DETR + Swin / DINOv2 demonstration

Start with recorded results, then optionally run existing models on authorized RGB
inputs. This notebook does not train models or access the reserved test set.

Install from the repository root using `python -m pip install -r method_b/requirements.txt`,
then launch `jupyter lab method_b/notebooks/demo.ipynb`. Select that environment's kernel.

Pipeline: RGB region → RF-DETR boxes → same-image crops → Swin-Tiny or DINOv2 labels.


In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import sys
import tempfile

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "method_b/README.md").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository")
print("Repository:", ROOT)
print("Kernel Python:", sys.executable)


## 1. Inspect the recorded RGB comparison (no weights or GPU)


In [ ]:
comparison = json.loads((ROOT / "results/rgb_comparison.json").read_text())
for name in ("swin", "dino"):
    result = comparison[name]
    print(f"{name}: {result['correct']}/{result['detected']} matched targets correct "
          f"({result['classification_accuracy_on_detected']:.1%}); "
          f"end-to-end {result['end_to_end_success']:.1%}")
print("Scope:", comparison["caveat"])


The detector saw 13/14 validation slides during original training. These are diagnostic
results on fixed detections, not an independent unseen-slide detector test.


In [ ]:
(ROOT / "outputs").mkdir(exist_ok=True)
OUTPUT = Path(tempfile.mkdtemp(prefix="notebook_", dir=ROOT / "outputs"))
subprocess.run([sys.executable, "-m", "method_b.evaluation.export_results",
                "--output", str(OUTPUT / "tables")], cwd=ROOT, check=True)
with (OUTPUT / "tables/detector_crop_comparison.csv").open(newline="") as stream:
    print(stream.read())
print("Generated Method B tables:", OUTPUT / "tables")


## 2. Classify one real RGB specimen (optional)

Retrieve authorized images and verified weights following [the data guide](../../docs/DATA.md).
Supply a Swin checkpoint listed in `config/checkpoints.json`. DINO also requires the
pinned local Meta source. Turn `RUN_CLASSIFIER` on only after setting the paths.
Outputs below are new predictions, not additional benchmark measurements.


In [ ]:
RUN_CLASSIFIER = False
MODEL = "swin"  # or "dino"
IMAGE = ROOT / "external/specimen.png"
CHECKPOINT = ROOT / "external/swin.pt"
DINO_SOURCE = ROOT / "external/dinov2"
DEVICE = "cpu"

if RUN_CLASSIFIER:
    command = [sys.executable, "-m", "method_b.classification.predict_categories", "--model", MODEL,
               "--checkpoint", str(CHECKPOINT), "--image", str(IMAGE),
               "--device", DEVICE, "--output", str(OUTPUT / "crop_prediction.json")]
    if MODEL == "dino":
        command += ["--dino-source", str(DINO_SOURCE)]
    subprocess.run(command, cwd=ROOT, check=True)
    prediction = json.loads((OUTPUT / "crop_prediction.json").read_text())
    print(prediction["predictions"])
else:
    print("Real classifier inference skipped; configure paths and enable RUN_CLASSIFIER.")


## 3. Detect a region and classify the boxes (optional)

Set a single-class RF-DETR 2XL checkpoint and its independently recorded SHA-256.
Use an image region, not an entire NDPI slide. The detector writes image-local xyxy
boxes. Both classifiers crop from that same RGB region, with 10% context, white
square padding, bicubic 224 resize and ImageNet normalization.


In [ ]:
RUN_PIPELINE = False
REGION = ROOT / "external/region.png"
DETECTOR = ROOT / "external/detector.pth"
DETECTOR_SHA256 = ""  # expected fingerprint from the trusted experiment record

if RUN_PIPELINE:
    if len(DETECTOR_SHA256) != 64:
        raise ValueError("Supply the recorded detector SHA-256 before inference")
    command = [sys.executable, "-m", "method_b.pipeline.detect_and_classify",
               "--image", str(REGION), "--detector-checkpoint", str(DETECTOR),
               "--detector-sha256", DETECTOR_SHA256,
               "--classifier-checkpoint", str(CHECKPOINT), "--model", MODEL,
               "--device", DEVICE, "--output", str(OUTPUT / "pipeline")]
    if MODEL == "dino":
        command += ["--dino-source", str(DINO_SOURCE)]
    subprocess.run(command, cwd=ROOT, check=True)
    print(json.loads((OUTPUT / "pipeline/classifications.json").read_text())["predictions"])
else:
    print("Region inference skipped; configure paths and enable RUN_PIPELINE.")


## Sources and next steps

- [Swin](https://arxiv.org/abs/2103.14030), torchvision implementation.
- [DINOv2](https://arxiv.org/abs/2304.07193), Meta implementation.
- [RF-DETR](https://github.com/roboflow/rf-detr), recorded 2XL dependency.
- [Spring microscopy code](https://github.com/RiceD2KLab/Smithsonian_fossil_Sp26).
- [Attribution](../../NOTICE.md), [full workflow commands](../../docs/WORKFLOWS.md),
  [evaluation definitions](../../docs/METHODS.md).

Full frozen-RGB replay also checks source tile hashes, float32 coordinates and all
saved Swin categories. A small region demo does not replace those checks.
